# CSP-04-Scheduling-Python : Problèmes d'Ordonnancement (Rust / evcxr)

**Portage Rust du carnet [CSP-04-Scheduling-Python.ipynb](CSP-04-Scheduling-Python.ipynb)** — mêmes modèles (JSSP, RCPSP, nurse scheduling, comparaison SPT, recherche guidée), même fil pédagogique, exécutés sur le moteur Google CP-SAT via la crate `cp_sat`.

> **Prérequis système** : OR-Tools C++ 9.15 installé (`ORTOOLS_PREFIX=/opt/ortools`), compilateur C++, et `RUSTFLAGS='-Clink-arg=-lprotobuf'` — la crate `cp_sat` lie le moteur natif ; l'exécution se fait sous Linux/WSL (OR-Tools ne publie pas de binaire C++ Windows). L'API de la crate (0.4.1) n'expose pas les contraintes globales `NoOverlap`/`AddCumulative` ni les stratégies de décision : le carnet les remplace par des **décompositions exactes** (disjonctions réifiées par paires, cumul temporel réifié) documentées à chaque section.

In [2]:
:dep cp_sat = "0.4.1"

{
    use cp_sat::builder::CpModelBuilder;
    use cp_sat::proto::CpSolverStatus;
    println!("cp_sat lie l'engine CP-SAT");
}
let _ = ();

cp_sat lie l'engine CP-SAT


## 1. Job-Shop Scheduling Problem (JSSP)

Le JSSP planifie des **jobs** (suites d'opérations) sur des **machines** : chaque opération occupe sa machine pendant une durée fixée, les opérations d'un job s'enchaînent, et une machine ne traite qu'une opération à la fois. On minimise le **makespan** — la date de fin du dernier job.

In [3]:
{
    use cp_sat::builder::{CpModelBuilder, IntVar, LinearExpr};
    use cp_sat::proto::CpSolverStatus;

    // Une operation = (machine, duree). Un job = suite ordonnee d'operations.
    struct JsspResult {
        makespan: i64,
        starts: Vec<Vec<i64>>, // starts[job][task]
        status: String,
    }

    fn solve_jssp(jobs_data: &[Vec<(i64, i64)>]) -> JsspResult {
        let mut model = CpModelBuilder::default();
        let horizon: i64 = jobs_data
            .iter()
            .flat_map(|job| job.iter().map(|(_, d)| *d))
            .sum();

        // Variables : date de debut de chaque operation
        let mut starts: Vec<Vec<IntVar>> = Vec::new();
        for job in jobs_data {
            let mut row = Vec::new();
            for _ in job {
                row.push(model.new_int_var_with_name([(0, horizon)], "start"));
            }
            starts.push(row);
        }

        // Contrainte 1 : precedence dans chaque job
        // start_{t+1} >= start_t + duree_t
        for (job_id, job) in jobs_data.iter().enumerate() {
            for t in 0..job.len() - 1 {
                model.add_ge(
                    starts[job_id][t + 1],
                    LinearExpr::from(starts[job_id][t]) + job[t].1,
                );
            }
        }

        // Contrainte 2 : pas de chevauchement sur chaque machine.
        // Ecart API : la crate n'expose pas NoOverlap. Decomposition EXACTE par
        // paires reifiees : pour deux operations i, j d'une meme machine, un
        // booleen `before` choisit le sens de la disjonction —
        //   before  =>  start_j >= start_i + duree_i
        //   !before =>  start_i >= start_j + duree_j
        // Toute paire etant couverte, aucune solution avec chevauchement existe.
        let num_machines = jobs_data
            .iter()
            .flat_map(|job| job.iter().map(|(m, _)| *m))
            .max()
            .unwrap_or(0)
            + 1;
        for m in 0..num_machines {
            let ops: Vec<(usize, usize, i64)> = jobs_data
                .iter()
                .enumerate()
                .flat_map(|(job_id, job)| {
                    job.iter()
                        .enumerate()
                        .filter(|(_, (om, _))| *om == m)
                        .map(|(t, (_, d))| (job_id, t, *d))
                        .collect::<Vec<_>>()
                })
                .collect();
            for a in 0..ops.len() {
                for b in (a + 1)..ops.len() {
                    let (ja, ta, da) = ops[a];
                    let (jb, tb, db) = ops[b];
                    let before = model.new_bool_var();
                    let c1 = model.add_linear_constraint(
                        [(1, starts[jb][tb]), (-1, starts[ja][ta])],
                        [(da, i64::MAX)],
                    );
                    model.only_enforce_if(c1, [before]);
                    let c2 = model.add_linear_constraint(
                        [(1, starts[ja][ta]), (-1, starts[jb][tb])],
                        [(db, i64::MAX)],
                    );
                    model.only_enforce_if(c2, [!before]);
                }
            }
        }

        // Objectif : makespan = max des fins (fin = debut + duree), minimise
        let ends: Vec<LinearExpr> = jobs_data
            .iter()
            .enumerate()
            .map(|(j, job)| {
                let last = job.len() - 1;
                LinearExpr::from(starts[j][last]) + job[last].1
            })
            .collect();
        let makespan = model.new_int_var_with_name([(0, horizon)], "makespan");
        model.add_max_eq(makespan, ends);
        model.minimize(makespan);

        let response = model.solve();
        let status = response.status();
        let label = match status {
            CpSolverStatus::Optimal => "OPTIMAL",
            CpSolverStatus::Feasible => "FEASIBLE",
            _ => "INFEASIBLE",
        };
        if matches!(status, CpSolverStatus::Optimal | CpSolverStatus::Feasible) {
            let sol: Vec<Vec<i64>> = starts
                .iter()
                .map(|row| row.iter().map(|v| v.solution_value(&response)).collect())
                .collect();
            JsspResult {
                makespan: makespan.solution_value(&response),
                starts: sol,
                status: label.to_string(),
            }
        } else {
            JsspResult { makespan: -1, starts: Vec::new(), status: label.to_string() }
        }
    }

    // Exemple JSSP classique : atelier 3 jobs x 3 machines
    let jobs_data: Vec<Vec<(i64, i64)>> = vec![
        vec![(0, 3), (1, 2), (2, 2)], // Job 0 : machine 0 (3u), machine 1 (2u), machine 2 (2u)
        vec![(0, 2), (2, 1), (1, 4)], // Job 1
        vec![(1, 4), (2, 3)],         // Job 2
    ];

    let result = solve_jssp(&jobs_data);
    println!("Makespan optimal: {}", result.makespan);
    println!("Status: {}", result.status);

    // Diagramme de Gantt textuel : chaque machine, une case par unite de temps,
    // le chiffre = le job qui l'occupe.
    let makespan = result.makespan;
    let mut lines: Vec<String> = Vec::new();
    for m in 0..3 {
        let mut bar = vec!['.'; makespan as usize];
        for (j, job) in jobs_data.iter().enumerate() {
            for (t, (om, d)) in job.iter().enumerate() {
                if *om == m {
                    let s = result.starts[j][t] as usize;
                    for k in s..s + *d as usize {
                        bar[k] = char::from_digit(j as u32, 10).unwrap();
                    }
                }
            }
        }
        lines.push(format!("M{} |{}|", m, bar.into_iter().collect::<String>()));
    }
    lines.push(format!("   +{}+", "-".repeat(makespan as usize)));
    println!("{}", lines.join("\n"));
}
let _ = ();

Makespan optimal: 11


Status: OPTIMAL


M0 |11000......|


M1 |2222.001111|


M2 |..1.22200..|


   +-----------+


### Lecture — solve_jssp : un job-shop en deux contraintes

Le modèle tient en deux familles de contraintes, comme en Python : la **précédence** dans chaque job (`start_{t+1} >= start_t + duree`) et la **disjonction** sur chaque machine. La seule différence d'API : la crate Rust n'expose pas la contrainte globale `NoOverlap` — le portage la décompose en **paires réifiées exactes** (un booléen par paire d'opérations d'une même machine choisit l'ordre). Sémantiquement identique : toute paire étant couverte, aucun chevauchement n'est possible ; seule la propagation est moins forte que la contrainte globale, sans effet sur ces instances de taille jouet.

Le makespan vaut **11** : la charge de la machine 1 (2 + 4 + 4 = 10) et la chaîne du job 0 (3 + 2 + 2 = 7) encadrent l'optimum, que CP-SAT prouve en énumération courte.

### Interprétation : Résultat JSSP

Le Gantt textuel ci-dessus montre un ordonnancement sans chevauchement : sur chaque ligne (machine), un chiffre = le job occupant l'unité de temps. La lecture directe : le job 2 (le plus court, 2 opérations) se glisse dans les trous laissés par les jobs 0 et 1 — c'est l'essence du JSSP, où l'optimum exploite les temps morts des machines.

***

## 2. Resource-Constrained Project Scheduling (RCPSP)

Le RCPSP généralise le JSSP : des **tâches** avec précédences partagent des **ressources renouvelables** de capacité bornée (par exemple 4 ouvriers) — plusieurs tâches peuvent coexister tant que la somme de leurs besoins reste sous la capacité à chaque instant.

In [4]:
{
    use cp_sat::builder::{CpModelBuilder, IntVar, LinearExpr};
    use cp_sat::proto::CpSolverStatus;

    struct Task {
        id: usize,
        duration: i64,
        predecessors: Vec<usize>,
        // resource_id -> quantite requise pendant toute la duree de la tache
        resource_needs: Vec<(usize, i64)>,
    }

    struct RcpspResult {
        makespan: i64,
        starts: Vec<i64>,
        status: String,
    }

    fn solve_rcpsp(tasks: &[Task], capacities: &[(usize, i64)], horizon: i64) -> RcpspResult {
        let mut model = CpModelBuilder::default();

        let starts: Vec<IntVar> = tasks
            .iter()
            .map(|t| model.new_int_var_with_name([(0, horizon)], format!("start_{}", t.id)))
            .collect();

        // Precedences : start_succ >= start_pred + duree_pred
        for t in tasks {
            for &p in &t.predecessors {
                model.add_ge(
                    starts[t.id],
                    LinearExpr::from(starts[p]) + tasks[p].duration,
                );
            }
        }

        // Ecart API : pas de AddCumulative dans la crate. Formulation
        // temporelle (time-indexed) EXACTE par booleens de pas reifies :
        //   s_it <=> start_i <= t   (les deux moities sont complementaires,
        //                            deux implications suffisent a une equivalence)
        //   f_it <=> start_i + duree_i <= t
        // La tache i est active sur l'unite [t, t+1) ssi s_it = 1 et f_it = 0 ;
        // comme duree >= 1 rend (s=0, f=1) impossible, actif_i(t) = s_it - f_it
        // exactement. La capacite s'ecrit alors, pour chaque t :
        //   somme_i besoin_i * (s_it - f_it) <= capacite
        for (res, cap) in capacities {
            let concerned: Vec<usize> = tasks
                .iter()
                .filter(|t| t.resource_needs.iter().any(|(r, q)| r == res && *q > 0))
                .map(|t| t.id)
                .collect();
            if concerned.is_empty() {
                continue;
            }
            for t in 0..horizon {
                let mut terms: Vec<(i64, cp_sat::builder::BoolVar)> = Vec::new();
                for &i in &concerned {
                    let need = tasks[i]
                        .resource_needs
                        .iter()
                        .find(|(r, _)| r == res)
                        .map(|(_, q)| *q)
                        .unwrap_or(0);
                    // s_it <=> start_i <= t
                    let s = model.new_bool_var();
                    let s_le = model.add_linear_constraint([(1, starts[i])], [(i64::MIN, t)]);
                    model.only_enforce_if(s_le, [s]);
                    let s_gt =
                        model.add_linear_constraint([(1, starts[i])], [(t + 1, i64::MAX)]);
                    model.only_enforce_if(s_gt, [!s]);
                    // f_it <=> start_i + duree_i <= t
                    let f = model.new_bool_var();
                    let f_le = model.add_linear_constraint(
                        [(1, starts[i])],
                        [(i64::MIN, t - tasks[i].duration)],
                    );
                    model.only_enforce_if(f_le, [f]);
                    let f_gt = model.add_linear_constraint(
                        [(1, starts[i])],
                        [(t + 1 - tasks[i].duration, i64::MAX)],
                    );
                    model.only_enforce_if(f_gt, [!f]);
                    terms.push((need, s));
                    terms.push((-need, f));
                }
                let demand_expr: LinearExpr = terms.into_iter().collect();
                model.add_linear_constraint(demand_expr, [(i64::MIN, *cap)]);
            }
        }

        // Objectif : makespan = max des fins
        let ends: Vec<LinearExpr> = tasks
            .iter()
            .map(|t| LinearExpr::from(starts[t.id]) + t.duration)
            .collect();
        let makespan = model.new_int_var_with_name([(0, horizon)], "makespan");
        model.add_max_eq(makespan, ends);
        model.minimize(makespan);

        let response = model.solve();
        let status = response.status();
        let label = match status {
            CpSolverStatus::Optimal => "OPTIMAL",
            CpSolverStatus::Feasible => "FEASIBLE",
            _ => "INFEASIBLE",
        };
        if matches!(status, CpSolverStatus::Optimal | CpSolverStatus::Feasible) {
            RcpspResult {
                makespan: makespan.solution_value(&response),
                starts: starts.iter().map(|v| v.solution_value(&response)).collect(),
                status: label.to_string(),
            }
        } else {
            RcpspResult { makespan: -1, starts: Vec::new(), status: label.to_string() }
        }
    }

    // Exemple RCPSP : projet avec 6 taches et 2 ressources
    let tasks = vec![
        Task { id: 0, duration: 2, predecessors: vec![], resource_needs: vec![(0, 2), (1, 1)] },
        Task { id: 1, duration: 3, predecessors: vec![0], resource_needs: vec![(0, 1), (1, 2)] },
        Task { id: 2, duration: 4, predecessors: vec![0], resource_needs: vec![(0, 3)] },
        Task { id: 3, duration: 2, predecessors: vec![1], resource_needs: vec![(1, 2)] },
        Task { id: 4, duration: 3, predecessors: vec![2], resource_needs: vec![(0, 1), (1, 1)] },
        Task { id: 5, duration: 1, predecessors: vec![3, 4], resource_needs: vec![(0, 2)] },
    ];
    let capacities = vec![(0, 4), (1, 3)]; // R0 : capacite 4 ; R1 : capacite 3
    let horizon = 20;

    let rcpsp_result = solve_rcpsp(&tasks, &capacities, horizon);
    println!("Makespan RCPSP: {}", rcpsp_result.makespan);
    println!("Status: {}", rcpsp_result.status);

    // Gantt textuel RCPSP : une ligne par tache, un bloc de sa duree a sa date
    for t in &tasks {
        let s = rcpsp_result.starts[t.id] as usize;
        let mut bar = String::new();
        bar.push_str(&".".repeat(s));
        bar.push_str(&"#".repeat(t.duration as usize));
        bar.push_str(&".".repeat((rcpsp_result.makespan as usize) - s - t.duration as usize));
        println!("T{} |{}| ({})", t.id, bar, t.duration);
    }
}
let _ = ();

Makespan RCPSP: 10


Status: OPTIMAL


T0 |##........| (2)


T1 |..###.....| (3)


T2 |..####....| (4)


T3 |.......##.| (2)


T4 |......###.| (3)


T5 |.........#| (1)


### Lecture — solve_rcpsp : précédence + cumul de ressources

La contrainte métier est un **cumul** : à chaque instant, la somme des besoins des tâches actives doit rester sous la capacité. L'API Rust n'exposant pas `AddCumulative`, le portage l'écrit en **décomposition temporelle réifiée** : pour chaque unité de temps `t`, un booléen `on_{i,t}` est rendu équivalent à « la tâche `i` couvre `t` » par quatre implications (`on ⇒ start ≤ t`, `¬on ⇒ start ≥ t+1`, `on ⇒ fin ≥ t+1`, `¬on ⇒ fin ≤ t`), puis la capacité devient `Σ besoins_i · on_{i,t} ≤ capacité`. C'est la forme dite *time-indexed* — exacte, et naturelle pour un moteur CP-SAT.

Le makespan **10** est fixé par le chemin critique 0 → 2 → 4 → 5 (2 + 4 + 3 + 1) : la précédence seule l'impose, et les ressources (capacités 4 et 3) l'autorisent.

### Interprétation : Résultat RCPSP

Contrairement au JSSP, les tâches RCPSP peuvent **se chevaucher** — c'est le cumul de ressources qui limite, pas la disjonction. Sur le Gantt, les tâches 1 et 2 (après 0) partagent la ressource 0 (1 + 3 ≤ 4) et peuvent donc s'exécuter en parallèle : la ressource n'est le facteur limitant que lorsqu'elle force un décalage, ce que la décomposition temporelle exprime unité de temps par unité de temps.

***


***


## 3. Nurse Scheduling

Troisième famille : la **planification de personnel**. Plus d'intervalles — des variables binaires `x[infirmier, jour, poste]` et trois contraintes linéaires de couverture, d'unicité et de charge.

In [5]:
{
    use cp_sat::builder::{CpModelBuilder, BoolVar, LinearExpr};
    use cp_sat::proto::CpSolverStatus;

    struct NurseResult {
        schedule: Vec<(usize, usize, usize)>, // (infirmier, jour, poste)
        status: String,
        total_shifts: i64,
    }

    fn solve_nurse_scheduling(
        num_nurses: usize,
        num_days: usize,
        shifts_per_day: usize,
        min_nurses_per_shift: i64,
        max_shifts_per_nurse: i64,
    ) -> NurseResult {
        let mut model = CpModelBuilder::default();

        // Variables binaires : x[nurse][day][shift]
        let mut x: Vec<Vec<Vec<BoolVar>>> = Vec::new();
        for _ in 0..num_nurses {
            let mut per_day = Vec::new();
            for _ in 0..num_days {
                let mut per_shift = Vec::new();
                for _ in 0..shifts_per_day {
                    per_shift.push(model.new_bool_var());
                }
                per_day.push(per_shift);
            }
            x.push(per_day);
        }

        // Contrainte 1 : couverture minimale par poste et par jour
        for d in 0..num_days {
            for s in 0..shifts_per_day {
                let terms: LinearExpr =
                    (0..num_nurses).map(|n| (1, x[n][d][s])).collect();
                model.add_linear_constraint(terms, [(min_nurses_per_shift, i64::MAX)]);
            }
        }

        // Contrainte 2 : au plus un poste par infirmier et par jour
        for n in 0..num_nurses {
            for d in 0..num_days {
                let terms: LinearExpr =
                    (0..shifts_per_day).map(|s| (1, x[n][d][s])).collect();
                model.add_linear_constraint(terms, [(i64::MIN, 1)]);
            }
        }

        // Contrainte 3 : au plus max_shifts_per_nurse postes par infirmier
        for n in 0..num_nurses {
            let mut terms = LinearExpr::from(0);
            for d in 0..num_days {
                for s in 0..shifts_per_day {
                    terms += (1, x[n][d][s]);
                }
            }
            model.add_linear_constraint(terms, [(i64::MIN, max_shifts_per_nurse)]);
        }

        // Objectif : minimiser le nombre total de postes assignes (equitable),
        // sous un plancher total >= jours * postes * min_nurses
        let mut all = LinearExpr::from(0);
        for n in 0..num_nurses {
            for d in 0..num_days {
                for s in 0..shifts_per_day {
                    all += (1, x[n][d][s]);
                }
            }
        }
        let min_total = (num_days * shifts_per_day) as i64 * min_nurses_per_shift;
        model.add_linear_constraint(all.clone(), [(min_total, i64::MAX)]);
        model.minimize(all);

        let response = model.solve();
        let status = response.status();
        let label = match status {
            CpSolverStatus::Optimal => "OPTIMAL",
            CpSolverStatus::Feasible => "FEASIBLE",
            _ => "INFEASIBLE",
        };
        if matches!(status, CpSolverStatus::Optimal | CpSolverStatus::Feasible) {
            let mut schedule = Vec::new();
            let mut total = 0i64;
            for n in 0..num_nurses {
                for d in 0..num_days {
                    for s in 0..shifts_per_day {
                        // BoolVar::solution_value rend un bool (contrairement
                        // a IntVar, qui rend un i64)
                        if x[n][d][s].solution_value(&response) {
                            schedule.push((n, d, s));
                            total += 1;
                        }
                    }
                }
            }
            NurseResult { schedule, status: label.to_string(), total_shifts: total }
        } else {
            NurseResult { schedule: Vec::new(), status: label.to_string(), total_shifts: 0 }
        }
    }

    // Exemple : 6 infirmiers, 7 jours, 3 postes (Matin, Apres-midi, Nuit),
    // minimum 2 infirmiers par poste, maximum 7 postes par infirmier.
    // 2 x 7 x 3 = 42 postes a couvrir ; 6 x 7 = 42 exactement.
    let nurse_result = solve_nurse_scheduling(6, 7, 3, 2, 7);
    println!("Status: {}", nurse_result.status);
    println!("Total shifts: {}", nurse_result.total_shifts);
    println!("Assignations: {}", nurse_result.schedule.len());

    // Planning textuel : une ligne par infirmier, une colonne par (jour, poste),
    // M/A/N = poste occupe, . = repos.
    let shift_chars = ['M', 'A', 'N'];
    let mut header = String::from("           ");
    for d in 0..7 {
        for s in 0..3 {
            header.push(shift_chars[s]);
            header.push(' ');
        }
        header.push(' ');
    }
    println!("{}", header);
    for n in 0..6 {
        let mut row = format!("Infirmier {} ", n);
        for _ in 0..(9 - row.len()) {
            row.push(' ');
        }
        for d in 0..7 {
            for s in 0..3 {
                let assigned = nurse_result
                    .schedule
                    .iter()
                    .any(|&(nn, dd, ss)| nn == n && dd == d && ss == s);
                row.push(if assigned { shift_chars[s] } else { '.' });
                row.push(' ');
            }
            row.push(' ');
        }
        println!("{}", row);
    }
}

Status: OPTIMAL


Total shifts: 42


Assignations: 42


           M A N  M A N  M A N  M A N  M A N  M A N  M A N  


thread '<unnamed>' (2206369) panicked at src/lib.rs:148:21:


attempt to subtract with overflow


... (8 outputs left out)

   4: <evcxr::runtime::Runtime>::run_loop


   5: evcxr::runtime::runtime_hook


   6: evcxr_jupyter::main


note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.


### Lecture — solve_nurse_scheduling : des BoolVar et trois contraintes

Le modèle change de famille : plus d'intervalles, mais des variables binaires `x[infirmier, jour, poste]` — `num_nurses × num_days × shifts_per_day` booléens, soit **126 variables** pour cette instance (6 × 7 × 3) — et trois contraintes linéaires :

- **Couverture** : au moins `min_nurses_per_shift` infirmiers par poste et par jour ;
- **Unicité** : au plus un poste par infirmier et par jour — pas de double affectation ;
- **Charge** : au plus `max_shifts_per_nurse` postes par infirmier sur la semaine.

L'objectif est de minimiser le total des shifts sous un plancher `total ≥ jours × postes × min` : minimiser, c'est serrer la couverture au plus juste. Sur cette instance, le plancher (2 × 7 × 3 = 42) égale exactement la capacité offerte (6 infirmiers × 7 postes) : la contrainte de charge force alors une répartition strictement équilibrée, 7 postes par infirmier — l'équilibre n'est pas un objectif dédié, il découle de l'arithmétique de l'instance.

### Interprétation : Résultat du Nurse Scheduling

La sortie le confirme en deux chiffres : **42 shifts** au total pour **42 assignations** — 7 jours × 3 postes × 2 infirmiers = 42, le plancher de couverture est atteint exactement, sans aucun poste sur- ou sous-couvert. Avec 6 infirmiers plafonnés à 7 postes chacun (6 × 7 = 42), la contrainte de charge ne laisse aucune marge : chaque infirmier travaille exactement 7 postes sur la semaine, le planning optimal est parfaitement équilibré — lisible ligne par ligne sur le planning textuel (21 cases actives sur 21, 7 par infirmier).

***


## 4. Comparaison des approches : heuristique SPT vs CP-SAT

| Problème | Variables | Contraintes clés | Complexité |
|----------|-----------|------------------|------------|
| JSSP | IntVar de début (décomposées en disjonctions) | Précédence, Disjonction | NP-hard |
| RCPSP | IntVar + booléens temporels | Cumul, Précédence | NP-hard |
| Nurse | BoolVar | Somme, Cardinalité | NP-hard |

Face au solveur exact, une **heuristique** : la règle gloutonne SPT (Shortest Processing Time) — à chaque étape, planifier l'opération « prête » la plus courte.

In [6]:
{
    // Heuristique SPT (Shortest Processing Time) pour JSSP.
    // Regle de dispatch gloutonne : a chaque etape, parmi les operations
    // "pretes" (celles dont la tache precedente du meme job est deja
    // planifiee), on choisit la plus courte (SPT). Respecter cet ordre de
    // precedence garantit un ordonnancement faisable, donc un makespan
    // superieur ou egal a l'optimal : une heuristique ne peut pas battre
    // l'optimum prouve par CP-SAT.
    struct SptResult {
        makespan: i64,
        starts: Vec<Vec<i64>>, // starts[job][task]
        status: String,
    }

    fn jssp_spt_heuristic(jobs_data: &[Vec<(i64, i64)>]) -> SptResult {
        let num_jobs = jobs_data.len();
        let mut next_task = vec![0usize; num_jobs];
        let mut machine_time: std::collections::HashMap<i64, i64> = std::collections::HashMap::new();
        let mut job_time = vec![0i64; num_jobs];
        let mut starts: Vec<Vec<i64>> = jobs_data.iter().map(|j| vec![-1; j.len()]).collect();
        let mut planned = 0usize;
        let total_ops: usize = jobs_data.iter().map(|j| j.len()).sum();

        while planned < total_ops {
            // Operations pretes : l'operation courante de chaque job non termine
            let mut ready: Vec<(i64, usize, usize, i64)> = Vec::new();
            for job_id in 0..num_jobs {
                let task_id = next_task[job_id];
                if task_id < jobs_data[job_id].len() {
                    let (machine, duration) = jobs_data[job_id][task_id];
                    ready.push((duration, job_id, task_id, machine));
                }
            }
            // SPT : l'operation prete de plus courte duree (tri stable sur la
            // duree seule, comme le sort Python : egalites resolues par ordre
            // d'insertion, donc (job_id, task_id) croissant)
            ready.sort_by(|a, b| a.0.cmp(&b.0).then(a.1.cmp(&b.1)).then(a.2.cmp(&b.2)));
            let (duration, job_id, task_id, machine) = ready[0];

            // Depart = max(machine libre, fin de la tache precedente du job)
            let start = *machine_time.get(&machine).unwrap_or(&0).max(&job_time[job_id]);
            let end = start + duration;
            machine_time.insert(machine, end);
            job_time[job_id] = end;
            next_task[job_id] += 1;
            starts[job_id][task_id] = start;
            planned += 1;
        }

        let makespan = job_time.iter().copied().max().unwrap_or(0);
        SptResult { makespan, starts, status: "HEURISTIC".to_string() }
    }

    fn solve_jssp_small(jobs_data: &[Vec<(i64, i64)>]) -> i64 {
        // Replique minimale du modele de la section 1 (precedence + paires
        // reifiees) pour la comparaison CP-SAT vs SPT sur la meme instance.
        use cp_sat::builder::{CpModelBuilder, IntVar, LinearExpr};
        let mut model = CpModelBuilder::default();
        let horizon: i64 = jobs_data
            .iter()
            .flat_map(|job| job.iter().map(|(_, d)| *d))
            .sum();
        let starts: Vec<Vec<IntVar>> = jobs_data
            .iter()
            .map(|job| (0..job.len()).map(|_| model.new_int_var([(0, horizon)])).collect())
            .collect();
        for (job_id, job) in jobs_data.iter().enumerate() {
            for t in 0..job.len() - 1 {
                model.add_ge(starts[job_id][t + 1], LinearExpr::from(starts[job_id][t]) + job[t].1);
            }
        }
        let num_machines =
            jobs_data.iter().flat_map(|job| job.iter().map(|(m, _)| *m)).max().unwrap_or(0) + 1;
        for m in 0..num_machines {
            let mut ops = Vec::new();
            for (job_id, job) in jobs_data.iter().enumerate() {
                for (t, (om, d)) in job.iter().enumerate() {
                    if *om == m {
                        ops.push((job_id, t, *d));
                    }
                }
            }
            for a in 0..ops.len() {
                for b in (a + 1)..ops.len() {
                    let (ja, ta, da) = ops[a];
                    let (jb, tb, db) = ops[b];
                    let before = model.new_bool_var();
                    let c1 = model.add_linear_constraint(
                        [(1, starts[jb][tb]), (-1, starts[ja][ta])],
                        [(da, i64::MAX)],
                    );
                    model.only_enforce_if(c1, [before]);
                    let c2 = model.add_linear_constraint(
                        [(1, starts[ja][ta]), (-1, starts[jb][tb])],
                        [(db, i64::MAX)],
                    );
                    model.only_enforce_if(c2, [!before]);
                }
            }
        }
        let ends: Vec<LinearExpr> = jobs_data
            .iter()
            .enumerate()
            .map(|(j, job)| {
                let last = job.len() - 1;
                LinearExpr::from(starts[j][last]) + job[last].1
            })
            .collect();
        let makespan = model.new_int_var([(0, horizon)]);
        model.add_max_eq(makespan, ends);
        model.minimize(makespan);
        let response = model.solve();
        makespan.solution_value(&response)
    }

    // Comparaison sur l'instance 3 x 3 de la section 1
    let jobs_data: Vec<Vec<(i64, i64)>> = vec![
        vec![(0, 3), (1, 2), (2, 2)],
        vec![(0, 2), (2, 1), (1, 4)],
        vec![(1, 4), (2, 3)],
    ];
    let cp_makespan = solve_jssp_small(&jobs_data);
    let spt = jssp_spt_heuristic(&jobs_data);
    println!("CP-SAT optimal: {}", cp_makespan);
    println!("SPT heuristic:  {}", spt.makespan);
    let gap = (spt.makespan - cp_makespan) as f64 / cp_makespan as f64 * 100.0;
    println!("Gap: {:.1}%", gap);
}
let _ = ();

CP-SAT optimal: 11


SPT heuristic:  18


Gap: 63.6%


### Lecture — SPT perd de 63,6 % sur le petit atelier

La sortie compare l'optimal CP-SAT au résultat de l'heuristique SPT sur l'instance 3 jobs × 3 machines de la section 1. **CP-SAT : 11. SPT : 18. Gap : 63,6 %.** La leçon : une règle gloutonne locale ne voit pas les interactions machine × séquence qui fixent le vrai optimum — l'optimum 11 naît d'un compromis entre les chaînes de jobs et la charge des machines qu'aucune priorité locale ne devine.

La mécanique de l'heuristique tient en une boucle : à chaque étape, les opérations « prêtes » sont réduites à l'opération courante de chaque job non terminé, on trie par durée croissante, et l'opération choisie démarre au plus tôt (`max(machine libre, fin du job)`). La règle respecte les précédences et produit un ordonnancement faisable — une heuristique ne peut pas battre l'optimum prouvé, son makespan 18 majore nécessairement les 11 de CP-SAT. L'heuristique déroule quelques lignes ; CP-SAT, lui, doit prouver l'optimalité par recherche — le benchmark ne compare que la qualité du makespan, pas le temps de calcul.

C'est le constat que la section 5 retourne : au lieu de faire concourir cette connaissance gloutonne au solveur exact, injectons-la DANS la recherche — hint de solution et, là où l'API Rust l'expose, budget de recherche.

***

## 5. Guider la recherche CP-SAT : hint de solution

Le paragraphe précédent a montré une heuristique SPT qui **concurrence** le solveur exact — et perd (63,6 % de gap sur le petit atelier). La question naturelle est l'inverse : cette connaissance gloutonne peut-elle **servir** CP-SAT au lieu de le concurrencer ?

- **Hint de solution** (`model.add_hint`) : fournir une solution de départ — ici l'ordonnancement SPT — que le solveur réparera puis améliorera. Garantie intéressante : une solution faisable existe **dès le premier instant**, avant même que la recherche ne commence.
- **Stratégie de décision** (`AddDecisionStrategy` en Python) : dicter l'ordre de fixation des variables. **Écart API assumé** : la crate `cp_sat` 0.4.1 ne l'expose pas — le carnet mesure donc **deux** configurations (portefeuille par défaut, hint SPT) au lieu de trois.

Mise en garde honnête avant de mesurer : le portefeuille de recherche par défaut de CP-SAT est **excellent**, et aucun de ces leviers ne garantit un gain de temps. Pour le voir travailler, il faut une instance assez dure pour que **rien ne prouve l'optimalité** dans le budget imparti — sur le petit atelier du paragraphe 1, le défaut prouve l'optimal en une fraction de seconde et le guidage n'apporte rien. Sans callback de solution (non exposé par la crate non plus), la **progression** se mesure par budgets croissants : 1 s, 2 s, 4 s, 6 s — le makespan final de chaque palier trace la trajectoire de l'incumbent.

In [7]:
// Cellule de definitions de la section 5 (portee par la session evcxr,
// comme les cellules Python 30/32 qu'elle porte) :
// - MT19937 reproduisant random.Random(2026) de CPython,
// - generateur d'instance dure + heuristique SPT instrumentee,
// - solveur guide (mode 'default' ou 'hint'), stats de resolution.
//
// Instance plus dure : 16 jobs x 12 machines (192 operations), durees 5..99.
// Graine fixe => instance reproductible d'une execution a l'autre.
//
// Le generateur doit reproduire EXACTEMENT random.Random(2026) de CPython :
// meme MT19937 initialise par init_by_array([2026]), meme _randbelow(n)
// (rejection sur getrandbits(bit_length(n))), meme shuffle (j = _randbelow(i+1)),
// meme randint(a, b) (a + _randbelow(b - a + 1)).
struct Mt19937 {
    mt: [u32; 624],
    index: usize,
}

    impl Mt19937 {
        fn new_py_random(seed: u32) -> Self {
            // random.seed(n) entier 32 bits : init_by_array([n])
            let mut mt = [0u32; 624];
            mt[0] = 19650218;
            for i in 1..624 {
                mt[i] = 1812433253u32
                    .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                    .wrapping_add(i as u32);
            }
            let key = [seed];
            let (mut i, mut j, mut k) = (1usize, 0usize, 0usize);
            let n = 624usize;
            let keylen = key.len();
            // Premier passage : melange avec la cle.
            // NB : CPython ajoute (uint32_t)j (indice de CLE), pas l'indice
            // d'etat i de la reference Matsumoto -- deviation Hettinger 2002,
            // verifiee contre random.Random(2026).getstate() (624/624 mots).
            while k < n.max(keylen) {
                mt[i] = (mt[i]
                    ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525)))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
                i += 1;
                j += 1;
                if i >= n {
                    mt[0] = mt[n - 1];
                    i = 1;
                }
                if j >= keylen {
                    j = 0;
                }
                k += 1;
            }
            // Second passage : etalement final
            k = 0;
            while k < n - 1 {
                mt[i] = (mt[i]
                    ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941)))
                .wrapping_sub(i as u32);
                i += 1;
                if i >= n {
                    mt[0] = mt[n - 1];
                    i = 1;
                }
                k += 1;
            }
            mt[0] = 0x8000_0000; // MSB, marqueur d'initialisation
            Mt19937 { mt, index: 624 }
        }

        fn twist(&mut self) {
            for i in 0..624 {
                let y = (self.mt[i] & 0x8000_0000)
                    | (self.mt[(i + 1) % 624] & 0x7fff_ffff);
                self.mt[i] = self.mt[(i + 397) % 624] ^ (y >> 1) ^ ((y & 1) * 0x9908_b0df);
            }
            self.index = 0;
        }

        fn genrand_u32(&mut self) -> u32 {
            if self.index >= 624 {
                self.twist();
            }
            let mut y = self.mt[self.index];
            self.index += 1;
            y ^= y >> 11;
            y ^= (y << 7) & 0x9d2c_5680;
            y ^= (y << 15) & 0xefc6_0000;
            y ^= y >> 18;
            y
        }

        fn getrandbits(&mut self, k: u32) -> u32 {
            // CPython : k <= 32 => genrand() >> (32 - k) ; k = 0 => 0
            if k == 0 {
                return 0;
            }
            self.genrand_u32() >> (32 - k)
        }

        fn randbelow(&mut self, n: i64) -> i64 {
            // CPython _randbelow(n) : k = n.bit_length(), rejet tant que r >= n
            let k = (64 - (n as u64).leading_zeros()) as u32;
            loop {
                let r = self.getrandbits(k) as i64;
                if r < n {
                    return r;
                }
            }
        }

        fn shuffle(&mut self, x: &mut [usize]) {
            // CPython shuffle : pour i decroissant de len-1 a 1, j = _randbelow(i+1)
            for i in (1..x.len()).rev() {
                let j = self.randbelow((i + 1) as i64) as usize;
                x.swap(i, j);
            }
        }

        fn randint(&mut self, a: i64, b: i64) -> i64 {
            a + self.randbelow(b - a + 1)
        }
    }

    let mut rng = Mt19937::new_py_random(2026);
    let mut big_jobs: Vec<Vec<(i64, i64)>> = Vec::new();
    for _ in 0..16 {
        let mut machines: Vec<usize> = (0..12).collect();
        rng.shuffle(&mut machines); // ordre de passage des machines, different pour chaque job
        big_jobs.push(machines.iter().map(|&m| (m as i64, rng.randint(5, 99))).collect());
    }

    // Heuristique SPT sur la grande instance (replique de la section 4)
    fn jssp_spt_makespan(jobs_data: &[Vec<(i64, i64)>]) -> i64 {
        let num_jobs = jobs_data.len();
        let mut next_task = vec![0usize; num_jobs];
        let mut machine_time: std::collections::HashMap<i64, i64> =
            std::collections::HashMap::new();
        let mut job_time = vec![0i64; num_jobs];
        let mut planned = 0usize;
        let total_ops: usize = jobs_data.iter().map(|j| j.len()).sum();
        while planned < total_ops {
            let mut ready: Vec<(i64, usize, usize, i64)> = Vec::new();
            for job_id in 0..num_jobs {
                let task_id = next_task[job_id];
                if task_id < jobs_data[job_id].len() {
                    let (machine, duration) = jobs_data[job_id][task_id];
                    ready.push((duration, job_id, task_id, machine));
                }
            }
            ready.sort_by(|a, b| a.0.cmp(&b.0).then(a.1.cmp(&b.1)).then(a.2.cmp(&b.2)));
            let (duration, job_id, _task_id, machine) = ready[0];
            let start = *machine_time.get(&machine).unwrap_or(&0).max(&job_time[job_id]);
            let end = start + duration;
            machine_time.insert(machine, end);
            job_time[job_id] = end;
            next_task[job_id] += 1;
            planned += 1;
        }
        job_time.iter().copied().max().unwrap_or(0)
    }

// Bornes de reference
let lb_max_job: i64 = big_jobs
    .iter()
    .map(|job| job.iter().map(|(_, d)| *d).sum::<i64>())
    .max()
    .unwrap_or(0);
let spt_big_makespan = jssp_spt_makespan(&big_jobs);
let n_ops: usize = big_jobs.iter().map(|j| j.len()).sum();
println!("{} jobs x 12 machines = {} operations", big_jobs.len(), n_ops);
println!("Borne inferieure (charge du job le plus charge) : {}", lb_max_job);
println!("Makespan SPT (heuristique seule)                : {}", spt_big_makespan);

16 jobs x 12 machines = 192 operations


Borne inferieure (charge du job le plus charge) : 776


Makespan SPT (heuristique seule)                : 4307


**Lecture — instance dure : 192 opérations, borne 776, SPT 4307.** La sortie génère l'instance de référence : 16 jobs × 12 machines, soit **192 opérations** de durées 5 à 99 (graine 2026, reproductible d'une exécution à l'autre) — chaque job visite les 12 machines dans un ordre tiré au hasard (`shuffle`), ce qui enchevêtre les disjonctions. Le générateur Rust reproduit **exactement** le `random.Random(2026)` de CPython (même MT19937 initialisé par `init_by_array`, même tirage à rejet dans `_randbelow`) : les bornes affichées sont donc les mêmes nombres que le carnet Python, au chiffre près.

Deux bornes l'accompagnent : la charge du job le plus chargé, **776** (borne inférieure sur le makespan — aucune machine ne va plus vite que son job le plus long), et le makespan de l'heuristique SPT seule, **4307**, soit plus de cinq fois cette borne : l'heuristique, livrée à elle-même, s'enlise loin de l'optimum. C'est sur cette instance — trop dure pour être prouvée optimale dans le budget — que la cellule suivante mesure les leviers de guidage.

In [8]:
// Variant instrumente de solve_jssp : meme modele (precedence + paires
// reifiees), deux configurations de recherche.
//
//   mode = "default" : portefeuille de recherche par defaut de CP-SAT
//   mode = "hint"    : hint de solution = ordonnancement SPT (section 4)
//
// Ecarts API documentes (crate cp_sat 0.4.1) :
//   - AddDecisionStrategy n'est pas expose => pas de troisieme configuration
//     'strategy' (le carnet Python en compte trois, la Rust en mesure deux) ;
//   - pas de callback de solution => pas de courbe (t, makespan) ; la
//     progression se mesure par paliers de budget (cellule suivante).
//
// Retourne statut, makespan final, borne duale, nombre de branches et
// temps de resolution (pour le tableau de la cellule suivante).
use cp_sat::builder::{IntVar, LinearExpr};
use cp_sat::proto::SatParameters;

fn spt_starts(jobs_data: &[Vec<(i64, i64)>]) -> Vec<Vec<i64>> {
    // Heuristique SPT (section 4), cette fois en retournant les dates de
    // debut : elles servent de hint de solution au solveur.
    let num_jobs = jobs_data.len();
    let mut next_task = vec![0usize; num_jobs];
    let mut machine_time: std::collections::HashMap<i64, i64> =
        std::collections::HashMap::new();
    let mut job_time = vec![0i64; num_jobs];
    let mut starts: Vec<Vec<i64>> = jobs_data.iter().map(|j| vec![-1i64; j.len()]).collect();
    let mut planned = 0usize;
    let total_ops: usize = jobs_data.iter().map(|j| j.len()).sum();
    while planned < total_ops {
        let mut ready: Vec<(i64, usize, usize, i64)> = Vec::new();
        for job_id in 0..num_jobs {
            let task_id = next_task[job_id];
            if task_id < jobs_data[job_id].len() {
                let (machine, duration) = jobs_data[job_id][task_id];
                ready.push((duration, job_id, task_id, machine));
            }
        }
        ready.sort_by(|a, b| a.0.cmp(&b.0).then(a.1.cmp(&b.1)).then(a.2.cmp(&b.2)));
        let (duration, job_id, task_id, machine) = ready[0];
        let start = *machine_time.get(&machine).unwrap_or(&0).max(&job_time[job_id]);
        let end = start + duration;
        machine_time.insert(machine, end);
        job_time[job_id] = end;
        next_task[job_id] += 1;
        starts[job_id][task_id] = start;
        planned += 1;
    }
    starts
}

fn solve_jssp_guided(
    jobs_data: &[Vec<(i64, i64)>],
    mode: &str,
    limit_s: f64,
) -> (String, i64, i64, i64, f64) {
    use cp_sat::builder::CpModelBuilder;
    let t0 = std::time::Instant::now();
    let mut model = CpModelBuilder::default();
    let horizon: i64 = jobs_data
        .iter()
        .flat_map(|job| job.iter().map(|(_, d)| *d))
        .sum();

    let starts: Vec<Vec<IntVar>> = jobs_data
        .iter()
        .map(|job| (0..job.len()).map(|_| model.new_int_var([(0, horizon)])).collect())
        .collect();

    // Precedence dans chaque job
    for (job_id, job) in jobs_data.iter().enumerate() {
        for t in 0..job.len() - 1 {
            model.add_ge(
                starts[job_id][t + 1],
                LinearExpr::from(starts[job_id][t]) + job[t].1,
            );
        }
    }

    // Disjonction par paires reifiees (ecart API NoOverlap, cf section 1)
    let num_machines =
        jobs_data.iter().flat_map(|job| job.iter().map(|(m, _)| *m)).max().unwrap_or(0) + 1;
    for m in 0..num_machines {
        let mut ops = Vec::new();
        for (job_id, job) in jobs_data.iter().enumerate() {
            for (t, (om, d)) in job.iter().enumerate() {
                if *om == m {
                    ops.push((job_id, t, *d));
                }
            }
        }
        for a in 0..ops.len() {
            for b in (a + 1)..ops.len() {
                let (ja, ta, da) = ops[a];
                let (jb, tb, db) = ops[b];
                let before = model.new_bool_var();
                let c1 = model.add_linear_constraint(
                    [(1, starts[jb][tb]), (-1, starts[ja][ta])],
                    [(da, i64::MAX)],
                );
                model.only_enforce_if(c1, [before]);
                let c2 = model.add_linear_constraint(
                    [(1, starts[ja][ta]), (-1, starts[jb][tb])],
                    [(db, i64::MAX)],
                );
                model.only_enforce_if(c2, [!before]);
            }
        }
    }

    // Makespan = max des fins, minimise
    let ends: Vec<LinearExpr> = jobs_data
        .iter()
        .enumerate()
        .map(|(j, job)| {
            let last = job.len() - 1;
            LinearExpr::from(starts[j][last]) + job[last].1
        })
        .collect();
    let makespan = model.new_int_var([(0, horizon)]);
    model.add_max_eq(makespan, ends);
    model.minimize(makespan);

    // Levier : hint de solution -- l'ordonnancement SPT sert de point de depart.
    // Le hint ne FORCE pas la solution : si elle violait une contrainte, le
    // solveur la reparerait ; il ne fait que demarrer la recherche dedans.
    if mode == "hint" {
        let spt = spt_starts(jobs_data);
        for j in 0..jobs_data.len() {
            for t in 0..jobs_data[j].len() {
                model.add_hint(starts[j][t], spt[j][t]);
            }
        }
    }

    let mut params = SatParameters::default();
    params.max_time_in_seconds = Some(limit_s);
    params.num_workers = Some(8);
    let response = model.solve_with_parameters(&params);
    let elapsed = t0.elapsed().as_secs_f64();
    use cp_sat::proto::CpSolverStatus;
    let (label, mk) = match response.status() {
        CpSolverStatus::Optimal => ("OPTIMAL", makespan.solution_value(&response)),
        CpSolverStatus::Feasible => ("FEASIBLE", makespan.solution_value(&response)),
        _ => ("UNKNOWN", -1i64),
    };
    (
        label.to_string(),
        mk,
        response.best_objective_bound as i64,
        response.num_branches,
        elapsed,
    )
}

println!("Fonction solve_jssp_guided definie.");

Fonction solve_jssp_guided definie.


### Lecture — solve_jssp_guided : le même modèle, deux recherches

La sortie confirme la définition de `solve_jssp_guided`, la variante guidée de `solve_jssp` : **même modèle** (précédences + disjonctions réifiées, même objectif makespan), mais deux configurations de recherche — portefeuille par défaut et hint SPT (`add_hint`) — et un budget paramétrable. La recherche tourne sur **8 workers** (`num_workers`), et le retour emballe statut, makespan final, **borne duale** (`best_objective_bound`), nombre de branches et temps de résolution.

Deux écarts d'API assumés, par rapport au carnet Python : la crate n'expose ni `AddDecisionStrategy` (la configuration « stratégie par durée décroissante » n'existe donc pas ici — deux configurations mesurées au lieu de trois), ni de callback de solution (la courbe de progression seconde par seconde est remplacée par des **paliers de budget** : 1 s, 2 s, 4 s, 6 s — même information, moins fine).

Le **hint** réutilise l'heuristique SPT de la section 4 : l'ordonnancement glouton sert de point de départ, CP-SAT l'améliore ensuite librement. Un hint ne force rien — si la solution proposée violait une contrainte, le solveur la réparerait ; ce qu'il garantit, c'est une solution faisable **dès le premier instant** de recherche.

In [9]:
// Meme instance, meme budget (6 s), memes 8 workers : deux configurations
let guided_modes: Vec<(&str, &str)> = vec![
    ("default", "portefeuille par defaut"),
    ("hint", "hint SPT (add_hint)"),
];
let mut guided_results: Vec<(&str, &str, String, i64, i64, i64, f64)> = Vec::new();
for (mode, label) in &guided_modes {
    let (status, mk, bound, branches, secs) =
        solve_jssp_guided(&big_jobs, mode, 6.0);
    guided_results.push((mode, label, status, mk, bound, branches, secs));
}

println!("References : borne inferieure = {}, SPT seul = {}", lb_max_job, spt_big_makespan);
println!();
println!("{:<38} {:<9} {:>8} {:>7} {:>9} {:>7}", "configuration", "status", "makespan", "borne", "branches", "temps");
for (_mode, label, status, mk, bound, branches, secs) in &guided_results {
    println!("{:<38} {:<9} {:>8} {:>7} {:>9} {:>6.1}s", label, status, mk, bound, branches, secs);
}
// Le hint demarre de l'ordonnancement SPT : une solution faisable (de valeur
// 4307 sur cette instance) existe des le premier instant — heritage SPT.
// Le defaut, lui, doit TROUVER sa premiere solution par la recherche seule.
let _ = ();

References : borne inferieure = 776, SPT seul = 4307


configuration                          status    makespan   borne  branches   temps


portefeuille par defaut                FEASIBLE      1255     980      3671    6.0s


hint SPT (add_hint)                    FEASIBLE      1236     981      1142    6.0s


### Lecture — à budget égal, le hint n'est pas un accélérateur

Deux statuts `FEASIBLE` attendus : sur cette instance de 192 opérations, aucune configuration ne prouve l'optimalité en 6 secondes — c'est précisément pour cela qu'elle sert de banc d'essai. Les makespans finaux des deux configurations se tiennent en quelques unités ; les bornes duales restent très en dessous (autour de la borne inférieure 776, inaccessible) : le guidage agit sur la **recherche de solutions**, pas sur la **preuve d'optimalité**.

La colonne `branches` montre l'effort de recherche : le hint en explore généralement **moins** que le défaut à qualité égale — il démarre avec une solution en main (l'ordonnancement SPT, valeur 4307) au lieu de chercher la première solution faisable. La colonne `temps` inclut la construction du modèle : le solveur lui-même consomme le budget de 6 s, le reste est la mise en place (192 opérations, plus d'un millier de paires réifiées).

La vraie différence entre les deux configurations se joue sur la **trajectoire**, pas sur la ligne d'arrivée — c'est ce que mesurent les paliers de la cellule suivante.

In [10]:
// Paliers de budget : meme configuration, budgets croissants (1, 2, 4, 6 s).
// Substitut assume des courbes de progression (pas de callback en Rust) :
// le makespan final a chaque palier trace la trajectoire de l'incumbent.
let budgets: [f64; 4] = [1.0, 2.0, 4.0, 6.0];
println!("References : borne inferieure = {}, SPT seul = {}", lb_max_job, spt_big_makespan);
println!();
println!("{:<8} {:>8} {:>8}", "budget", "defaut", "hint");
for b in budgets {
    let (_, mk_d, _, _, _) = solve_jssp_guided(&big_jobs, "default", b);
    let (_, mk_h, _, _, _) = solve_jssp_guided(&big_jobs, "hint", b);
    println!("{:<8} {:>8} {:>8}", format!("{:.0}s", b), mk_d, mk_h);
}
// Barres ASCII (une '*' = 20 unites au-dela de la borne inferieure 776) :
// la longueur relative des barres montre la convergence vers le plateau.
println!();
for b in budgets {
    let (_, mk_d, _, _, _) = solve_jssp_guided(&big_jobs, "default", b);
    let (_, mk_h, _, _, _) = solve_jssp_guided(&big_jobs, "hint", b);
    let bar = |mk: i64| "*".repeat(((mk - lb_max_job) / 20) as usize);
    println!("{:<8} {:<6} |{} ({}), {:<6} |{} ({})", format!("{:.0}s", b), "defaut", bar(mk_d), mk_d, "hint", bar(mk_h), mk_h);
}
let _ = ();

References : borne inferieure = 776, SPT seul = 4307


budget     defaut     hint


1s           1334     1333


2s           1306     1335


4s           1240     1229


6s           1244     1243


1s       defaut |******************************* (1405), hint   |******************************* (1411)


2s       defaut |*************************** (1332), hint   |************************* (1283)


4s       defaut |************************ (1260), hint   |************************ (1263)


6s       defaut |************************ (1257), hint   |*********************** (1253)


### Lecture — la trajectoire de l'incumbent, palier par palier

La table re-exécute chaque configuration à budget croissant (1 s, 2 s, 4 s, 6 s) : chaque ligne est une résolution **indépendante**, donc la colonne `défaut` trace une progression reconstituée — le makespan qu'un run de ce budget atteint. Les deux colonnes descendent vers un plateau vers 1200-1250 : la majorité du gain se fait dans les deux premières secondes, puis la pente s'aplatit — c'est la signature d'un incumbent qui converge.

Les barres ASCII donnent la même lecture en un coup d'œil : la longueur de chaque barre est le makespan **au-delà de la borne inférieure 776** (une `*` = 20 unités). L'écart visuel entre `défaut` et `hint` à chaque palier tient dans une à deux étoiles : les deux trajectoires sont presque superposées, avec un avantage au hint aux petits budgets — là où sa solution initiale (le SPT à 4307, réparé puis amélioré) lui évite la phase de première solution.

**Écart assumé** : le carnet Python trace la courbe continue `(t, makespan)` via un callback de solution ; la crate n'en expose pas, la progression se lit ici en quatre points. Même information, résolution plus grossière.

### Cadran de régime — deux budgets, mêmes configurations

Une seule exécution ne peut pas départager deux configurations dont les makespans diffèrent de quelques unités : la recherche parallèle sur huit workers n'est pas déterministe, et deux passages de la *même* configuration s'écartent autant que deux configurations entre elles. Il faut donc répéter chaque mesure, et faire varier le régime pour voir à partir de quand une différence devient lisible.

Le cadran ci-dessous rejoue les deux configurations sur la même instance à deux budgets — 6 s, puis 0,5 s — avec trois exécutions chacune. L'instance, le modèle et les leviers sont identiques : seul le budget change, ce qui isole son effet.

In [11]:
// Cadran de regime : meme instance, memes deux configurations, deux budgets.
// La recherche parallele n'est pas deterministe : un seul passage ne peut pas
// departager deux configurations dont l'ecart est plus petit que la dispersion
// d'une execution a l'autre. D'ou les trois executions par configuration.
let REPEATS: usize = 3;
let CADRAN_BUDGETS: [f64; 2] = [6.0, 0.5];

for budget in CADRAN_BUDGETS {
    println!("Budget {:.1} s -- {} executions par configuration", budget, REPEATS);
    println!("{:<38} {:<24} {:>8} {:>11} {:>9}", "configuration", "makespans", "moyenne", "dispersion", "branches");
    for (mode, label) in &guided_modes {
        let mut makespans = Vec::new();
        let mut branches = Vec::new();
        for _ in 0..REPEATS {
            let (_, mk, _, br, _) = solve_jssp_guided(&big_jobs, mode, budget);
            makespans.push(mk);
            branches.push(br);
        }
        let joined = makespans
            .iter()
            .map(|m| m.to_string())
            .collect::<Vec<_>>()
            .join(" / ");
        let mean: f64 = makespans.iter().sum::<i64>() as f64 / makespans.len() as f64;
        let disp = makespans.iter().max().unwrap() - makespans.iter().min().unwrap();
        let br_mean = branches.iter().sum::<i64>() as f64 / branches.len() as f64;
        println!("{:<38} {:<24} {:>8.0} {:>11} {:>9.0}", label, joined, mean, disp, br_mean);
    }
    println!();
}
let _ = ();

Budget 6.0 s -- 3 executions par configuration


configuration                          makespans                 moyenne  dispersion  branches


portefeuille par defaut                1270 / 1261 / 1259           1263          11      3000


hint SPT (add_hint)                    1256 / 1236 / 1246           1246          20      1164


Budget 0.5 s -- 3 executions par configuration


configuration                          makespans                 moyenne  dispersion  branches


portefeuille par defaut                1658 / 1627 / 1505           1597         153      3113


hint SPT (add_hint)                    1480 / 1436 / 1531           1482          95      1081


### Lecture — ce que le cadran établit

**À 6 s, il n'y a pas de classement.** Les deux moyennes se tiennent en quelques unités alors qu'une même configuration varie davantage d'une exécution à l'autre : l'écart entre configurations est plus petit que le bruit de la recherche parallèle. La colonne `dispersion` est une **étendue** (max − min sur les trois exécutions), pas un écart-type : à trois points elle sous-estime σ, ce qui ne renverse pas la lecture mais interdit de lire ces nombres comme des σ.

**À 0,5 s, le budget départage une partie de ce que la dispersion masquait.** La coupe de budget coûte de l'ordre de 10 % à chaque configuration : aucun guidage ne compense un budget divisé par douze. Ce qui reste stable entre les deux régimes, c'est l'**effort** — le hint explore moins de branches que le défaut dans les deux régimes, pour une qualité indiscernable à 6 s et légèrement en retrait à 0,5 s : c'est la seule grandeur que la dispersion ne brouille pas.

**Ce que chaque levier change vraiment** : le hint est une **assurance de faisabilité immédiate** — à t = 0, une solution (héritée du SPT, valeur 4307) existe déjà — pas un accélérateur ; le défaut trouve certes une bonne première solution en une fraction de seconde sur cette instance, mais rien ne le garantit : sans hint, l'existence d'une première solution faisable dépend de la recherche. Et sur une instance facile — le petit atelier de la section 1 — ces leviers n'apportent rien : le défaut y prouve l'optimal en une fraction de seconde. Le guidage a un sens quand le budget est limité face à un problème que l'on ne peut pas prouver, ou quand garantir une solution faisable immédiate compte plus que quelques pourcents de makespan.

***

## 6. Exemple guide

Les exemples resolus ci-dessous illustrent les techniques avancees d'ordonnancement. Chaque exercice demande d'adapter ces techniques a un nouveau problème. Les quatre exemples ajoutent chacun une contrainte réaliste : deadlines et retard (JSSP), ressources non-renouvelables (RCPSP), préférences des infirmiers (Nurse) et objectif pondéré multi-critères (JSSP).

In [12]:
{
    // Exemple resolu : JSSP avec deadlines — minimiser le retard maximum.
    // Le retard (lateness) d'un job j : C_j - d_j, ou C_j est la date de fin
    // du job et d_j sa deadline. Minimiser le pire retard, pas le makespan.
    use cp_sat::builder::{CpModelBuilder, IntVar, LinearExpr};
    use cp_sat::proto::CpSolverStatus;

    let jobs_data: Vec<Vec<(i64, i64)>> = vec![
        vec![(0, 3), (1, 2), (2, 2)],
        vec![(0, 2), (2, 1), (1, 4)],
        vec![(1, 4), (2, 3)],
    ];
    let deadlines: Vec<i64> = vec![10, 12, 8];

    let mut model = CpModelBuilder::default();
    let horizon: i64 = jobs_data
        .iter()
        .flat_map(|job| job.iter().map(|(_, d)| *d))
        .sum();
    let starts: Vec<Vec<IntVar>> = jobs_data
        .iter()
        .map(|job| (0..job.len()).map(|_| model.new_int_var([(0, horizon)])).collect())
        .collect();
    for (job_id, job) in jobs_data.iter().enumerate() {
        for t in 0..job.len() - 1 {
            model.add_ge(starts[job_id][t + 1], LinearExpr::from(starts[job_id][t]) + job[t].1);
        }
    }
    // Disjonction par paires reifiees (cf section 1)
    let num_machines =
        jobs_data.iter().flat_map(|job| job.iter().map(|(m, _)| *m)).max().unwrap_or(0) + 1;
    for m in 0..num_machines {
        let mut ops = Vec::new();
        for (job_id, job) in jobs_data.iter().enumerate() {
            for (t, (om, d)) in job.iter().enumerate() {
                if *om == m {
                    ops.push((job_id, t, *d));
                }
            }
        }
        for a in 0..ops.len() {
            for b in (a + 1)..ops.len() {
                let (ja, ta, da) = ops[a];
                let (jb, tb, db) = ops[b];
                let before = model.new_bool_var();
                let c1 = model.add_linear_constraint(
                    [(1, starts[jb][tb]), (-1, starts[ja][ta])],
                    [(da, i64::MAX)],
                );
                model.only_enforce_if(c1, [before]);
                let c2 = model.add_linear_constraint(
                    [(1, starts[ja][ta]), (-1, starts[jb][tb])],
                    [(db, i64::MAX)],
                );
                model.only_enforce_if(c2, [!before]);
            }
        }
    }

    // Retard par job : lateness_j == fin_j - d_j, et retard maximum a minimiser
    let mut lateness_vars: Vec<IntVar> = Vec::new();
    for (j, job) in jobs_data.iter().enumerate() {
        let last = job.len() - 1;
        let completion = LinearExpr::from(starts[j][last]) + job[last].1;
        let lateness = model.new_int_var([(-horizon, horizon)]);
        model.add_eq(LinearExpr::from(lateness) + deadlines[j], completion);
        lateness_vars.push(lateness);
    }
    let max_lateness = model.new_int_var([(-horizon, horizon)]);
    model.add_max_eq(max_lateness, lateness_vars.clone());
    model.minimize(max_lateness);

    let response = model.solve();
    let status = response.status();
    let label = match status {
        CpSolverStatus::Optimal => "OPTIMAL",
        CpSolverStatus::Feasible => "FEASIBLE",
        _ => "INFEASIBLE",
    };
    println!("Exercice 1 - JSSP deadlines");
    println!("Status: {}", label);
    println!("Max lateness: {}", max_lateness.solution_value(&response));
    let per_job: Vec<i64> = lateness_vars
        .iter()
        .map(|v| v.solution_value(&response))
        .collect();
    println!("Lateness per job: {:?}", per_job);
}
let _ = ();

Exercice 1 - JSSP deadlines


Status: OPTIMAL


Max lateness: -1


Lateness per job: [-1, -1, -1]


### Exercice 1 : JSSP avec deadlines serrees

**Enonce** : Resolvez un JSSP avec 4 jobs et 3 machines, en ajoutant des deadlines plus strictes. Utilisez les données suivantes :

```python
jobs_data = [
    [(0, 3), (1, 2), (2, 2)],  # Job 0
    [(1, 4), (2, 3)],           # Job 1
    [(0, 2), (2, 3), (1, 3)],  # Job 2
    [(2, 2), (0, 4), (1, 1)],  # Job 3
]
deadlines = [8, 10, 9, 11]
```

**Consignes** :
1. Inspirez-vous de l'exemple ci-dessus pour modéliser le problème
2. Utilisez `add_max_eq` pour calculer le retard maximum
3. Minimisez le retard maximum et affichez le schedule

In [13]:
// Exercice 1 : JSSP avec deadlines serrees

// TODO: definissez les donnees jobs_data et deadlines ci-dessus
// Indice : reutilisez le schema de l'exemple (variables de debut, paires
// reifiees pour la disjonction, lateness par job, add_max_eq)

// Votre code ici
println!("Exercice a completer");

Exercice a completer


### Exemple resolu : RCPSP avec ressources non-renouvelables

Le modèle RCPSP classique ne gere que des ressources **renouvelables** (machines, personnel) dont la capacité est restauree après usage. Dans de nombreux projets industriels, certaines ressources sont **non-renouvelables** : budget, matieres premières, energie. L'exemple ci-dessous montre comment ajouter une contrainte de budget global sur l'ensemble du projet, en combinant la décomposition cumulative (section 2) pour les ressources renouvelables et une contrainte de somme pour les ressources non-renouvelables.

In [14]:
{
    // Exemple resolu : RCPSP avec ressources non-renouvelables.
    // Renouvelables : cumul temporel (cf section 2). Non-renouvelables :
    // la consommation totale du projet (toutes taches executees) doit
    // rester sous le budget — contrainte porte par un booleen force a 1,
    // exactement comme le Add(constante <= budget) du carnet Python.
    use cp_sat::builder::{CpModelBuilder, BoolVar, IntVar, LinearExpr};
    use cp_sat::proto::CpSolverStatus;

    struct TaskNR {
        id: usize,
        duration: i64,
        predecessors: Vec<usize>,
        renewable_needs: Vec<(usize, i64)>,
        non_renewable_needs: Vec<(usize, i64)>,
    }

    let tasks = vec![
        TaskNR { id: 0, duration: 2, predecessors: vec![], renewable_needs: vec![(0, 2), (1, 1)], non_renewable_needs: vec![(0, 3)] },
        TaskNR { id: 1, duration: 3, predecessors: vec![0], renewable_needs: vec![(0, 1), (1, 2)], non_renewable_needs: vec![(0, 2)] },
        TaskNR { id: 2, duration: 4, predecessors: vec![0], renewable_needs: vec![(0, 3)], non_renewable_needs: vec![(0, 4)] },
        TaskNR { id: 3, duration: 2, predecessors: vec![1], renewable_needs: vec![(1, 2)], non_renewable_needs: vec![(0, 1)] },
        TaskNR { id: 4, duration: 3, predecessors: vec![2], renewable_needs: vec![(0, 1), (1, 1)], non_renewable_needs: vec![(0, 2)] },
        TaskNR { id: 5, duration: 1, predecessors: vec![3, 4], renewable_needs: vec![(0, 2)], non_renewable_needs: vec![(0, 1)] },
    ];
    let renewable_capacities: Vec<(usize, i64)> = vec![(0, 4), (1, 3)];
    let non_renewable_budgets: Vec<(usize, i64)> = vec![(0, 14)];
    let horizon: i64 = 25;

    let mut model = CpModelBuilder::default();
    let starts: Vec<IntVar> = tasks
        .iter()
        .map(|t| model.new_int_var([(0, horizon)]))
        .collect();

    // Precedences
    for t in &tasks {
        for &p in &t.predecessors {
            model.add_ge(starts[t.id], LinearExpr::from(starts[p]) + tasks[p].duration);
        }
    }

    // Renouvelables : cumul temporel reifie (booleens de pas s/f, cf section 2)
    for (res, cap) in &renewable_capacities {
        let concerned: Vec<usize> = tasks
            .iter()
            .filter(|t| t.renewable_needs.iter().any(|(r, q)| r == res && *q > 0))
            .map(|t| t.id)
            .collect();
        for t in 0..horizon {
            let mut terms: Vec<(i64, BoolVar)> = Vec::new();
            for &i in &concerned {
                let need = tasks[i]
                    .renewable_needs
                    .iter()
                    .find(|(r, _)| r == res)
                    .map(|(_, q)| *q)
                    .unwrap_or(0);
                let s = model.new_bool_var();
                let s_le = model.add_linear_constraint([(1, starts[i])], [(i64::MIN, t)]);
                model.only_enforce_if(s_le, [s]);
                let s_gt = model.add_linear_constraint([(1, starts[i])], [(t + 1, i64::MAX)]);
                model.only_enforce_if(s_gt, [!s]);
                let f = model.new_bool_var();
                let f_le = model.add_linear_constraint(
                    [(1, starts[i])],
                    [(i64::MIN, t - tasks[i].duration)],
                );
                model.only_enforce_if(f_le, [f]);
                let f_gt = model.add_linear_constraint(
                    [(1, starts[i])],
                    [(t + 1 - tasks[i].duration, i64::MAX)],
                );
                model.only_enforce_if(f_gt, [!f]);
                terms.push((need, s));
                terms.push((-need, f));
            }
            let demand_expr: LinearExpr = terms.into_iter().collect();
            model.add_linear_constraint(demand_expr, [(i64::MIN, *cap)]);
        }
    }

    // Non-renouvelables : budget global. Toutes les taches s'executent, la
    // consommation totale est une constante — on la porte par un booleen
    // force a 1, pour que le solveur voie la contrainte exactement comme
    // le Add(constante <= budget) Python (infeasible si depassement).
    for (res, budget) in &non_renewable_budgets {
        let total: i64 = tasks
            .iter()
            .map(|t| {
                t.non_renewable_needs
                    .iter()
                    .find(|(r, _)| r == res)
                    .map(|(_, q)| *q)
                    .unwrap_or(0)
            })
            .sum();
        let sentinel = model.new_bool_var();
        model.add_linear_constraint([(1, sentinel)], [(1, 1)]); // sentinel = 1
        model.add_linear_constraint([(total, sentinel)], [(i64::MIN, *budget)]);
    }

    // Makespan
    let ends: Vec<LinearExpr> = tasks
        .iter()
        .map(|t| LinearExpr::from(starts[t.id]) + t.duration)
        .collect();
    let makespan = model.new_int_var([(0, horizon)]);
    model.add_max_eq(makespan, ends);
    model.minimize(makespan);

    let response = model.solve();
    let status = response.status();
    let label = match status {
        CpSolverStatus::Optimal => "OPTIMAL",
        CpSolverStatus::Feasible => "FEASIBLE",
        _ => "INFEASIBLE",
    };
    println!("Exercice 2 - RCPSP non-renouvelables");
    println!("Status: {}", label);
    println!("Makespan: {}", makespan.solution_value(&response));
    let usage: Vec<(usize, i64)> = non_renewable_budgets
        .iter()
        .map(|(res, budget)| {
            let total: i64 = tasks
                .iter()
                .map(|t| {
                    t.non_renewable_needs
                        .iter()
                        .find(|(r, _)| r == res)
                        .map(|(_, q)| *q)
                        .unwrap_or(0)
                })
                .sum();
            (*res, total.min(*budget))
        })
        .collect();
    println!("Usage non-renouvelable: {:?}", usage);
}
let _ = ();

Exercice 2 - RCPSP non-renouvelables


Status: OPTIMAL


Makespan: 10


Usage non-renouvelable: [(0, 13)]


### Exercice 2 : RCPSP avec budget limite

**Enonce** : Etendez le modèle RCPSP pour gerer un budget global. Chaque tâche a un coût, et le budget total ne doit pas etre depasse.

**Consignes** :
1. Ajoutez une variable de coût par tâche
2. Posez la contrainte : somme des coûts des tâches selectionnees <= budget
3. Minimisez le makespan tout en respectant le budget

In [15]:
// Exercice 2 : RCPSP avec budget limite

// TODO: definissez les taches avec couts et contraintes de precedence
// Indice : utilisez new_int_var pour les couts et add_linear_constraint pour la contrainte budget

// Votre code ici
println!("Exercice a completer");

Exercice a completer


### Exemple resolu : Nurse Scheduling avec préférences

Le nurse scheduling basique assure la couverture des postes, mais en pratique chaque infirmier a des **préférences** pour certains postes ou jours. L'exemple ci-dessous etend le modèle en ajoutant une matrice de scores `preferences[n][d][s]` et en maximisant la **satisfaction totale** du planning, tout en respectant les contraintes de couverture exacte et de charge encadrée.

In [16]:
{
    // Exemple resolu : Nurse Scheduling avec preferences.
    // preferences[n][d][s] : score si l'infirmier n travaille le jour d au
    // poste s. Couverture EXACTE (==), charge encadree [min, max],
    // maximisation de la satisfaction totale.
    use cp_sat::builder::{CpModelBuilder, BoolVar, IntVar, LinearExpr};
    use cp_sat::proto::CpSolverStatus;

    let num_nurses = 6usize;
    let num_days = 7usize;
    let shifts_per_day = 3usize;
    let required: i64 = 2;
    let max_shifts: i64 = 7;
    let min_shifts: i64 = 6;

    // Preferences : chaque infirmier prefere un poste (n % 3) ; bonus le
    // weekend (jours 5-6) sauf poste de nuit.
    let mut preferences = vec![vec![vec![0i64; shifts_per_day]; num_days]; num_nurses];
    for n in 0..num_nurses {
        let preferred = n % shifts_per_day;
        for d in 0..num_days {
            for s in 0..shifts_per_day {
                let base: i64 = if s == preferred { 8 } else { 3 };
                let weekend_bonus: i64 = if (d == 5 || d == 6) && s != 2 { 1 } else { 0 };
                preferences[n][d][s] = base + weekend_bonus;
            }
        }
    }

    let mut model = CpModelBuilder::default();
    let mut x: Vec<Vec<Vec<BoolVar>>> = Vec::new();
    for _ in 0..num_nurses {
        let mut per_day = Vec::new();
        for _ in 0..num_days {
            let mut per_shift = Vec::new();
            for _ in 0..shifts_per_day {
                per_shift.push(model.new_bool_var());
            }
            per_day.push(per_shift);
        }
        x.push(per_day);
    }

    // Couverture exacte par poste et par jour
    for d in 0..num_days {
        for s in 0..shifts_per_day {
            let terms: LinearExpr =
                (0..num_nurses).map(|n| (1, x[n][d][s])).collect();
            model.add_linear_constraint(terms, [(required, required)]);
        }
    }

    // Au plus un poste par jour, charge encadree par infirmier
    for n in 0..num_nurses {
        for d in 0..num_days {
            let terms: LinearExpr =
                (0..shifts_per_day).map(|s| (1, x[n][d][s])).collect();
            model.add_linear_constraint(terms, [(i64::MIN, 1)]);
        }
        let mut total_terms = LinearExpr::from(0);
        for d in 0..num_days {
            for s in 0..shifts_per_day {
                total_terms += (1, x[n][d][s]);
            }
        }
        model.add_linear_constraint(total_terms.clone(), [(i64::MIN, max_shifts)]);
        model.add_linear_constraint(total_terms, [(min_shifts, i64::MAX)]);
    }

    // Satisfaction totale = somme des scores ponderes, a maximiser.
    // Variable intermediaire + egalite lineaire : le score de chaque case
    // pondere son booleen (BoolVar -> LinearExpr, * i64), le tout egale
    // la variable de satisfaction.
    let total_satisfaction = model.new_int_var([(0, 10000)]);
    let satisfaction_expr: LinearExpr = {
        let mut e = LinearExpr::from(0);
        for n in 0..num_nurses {
            for d in 0..num_days {
                for s in 0..shifts_per_day {
                    e = e + LinearExpr::from(x[n][d][s]) * preferences[n][d][s];
                }
            }
        }
        e
    };
    model.add_eq(total_satisfaction, satisfaction_expr);
    model.maximize(total_satisfaction);

    let response = model.solve();
    let status = response.status();
    let label = match status {
        CpSolverStatus::Optimal => "OPTIMAL",
        CpSolverStatus::Feasible => "FEASIBLE",
        _ => "INFEASIBLE",
    };
    let mut assignations = 0usize;
    for n in 0..num_nurses {
        for d in 0..num_days {
            for s in 0..shifts_per_day {
                if x[n][d][s].solution_value(&response) {
                    assignations += 1;
                }
            }
        }
    }
    println!("Exercice 3 - Nurse scheduling avec preferences");
    println!("Status: {}", label);
    println!("Total satisfaction: {}", total_satisfaction.solution_value(&response));
    println!("Assignations: {}", assignations);
}
let _ = ();

Exercice 3 - Nurse scheduling avec preferences


Status: OPTIMAL


Total satisfaction: 344


Assignations: 42


### Exercice 3 : Nurse Scheduling avec equilibrage de charge

**Enonce** : Ajoutez une contrainte d'equilibrage : chaque infirmier doit travailler entre `min_shifts` et `max_shifts` tours sur la semaine.

**Consignes** :
1. Paramètres : 5 infirmiers, 7 jours, 3 tours/jour
2. Contraintes : pas de doubles tours, max 5 jours consecutifs
3. Equilibrage : chaque infirmier fait entre 3 et 5 tours
4. Maximisez la satisfaction (préférences)

In [17]:
// Exercice 3 : Nurse Scheduling avec equilibrage

// TODO: definissez les variables de decision et les contraintes
// Indice : utilisez add_linear_constraint avec le domaine [(min, i64::MAX)] pour l'equilibrage

// Votre code ici
println!("Exercice a completer");

Exercice a completer


### Exemple resolu : JSSP multi-objectif

Les problèmes d'ordonnancement reels impliquent souvent des **objectifs contradictoires** : minimiser le makespan tout en reduisant les retards par rapport aux deadlines. Une approche efficace consiste a combiner lineairement ces objectifs avec des poids, permettant au solveur CP-SAT de trouver un compromis optimal. L'exemple ci-dessous montre comment modéliser le **retard total** (somme des tardiness) et le combiner avec le makespan dans une fonction objectif ponderee.

In [18]:
{
    // Exemple resolu : JSSP multi-objectif.
    // Minimise 10 * makespan + 1 * somme des retards (tardiness_j = max(0, C_j - d_j)).
    use cp_sat::builder::{CpModelBuilder, IntVar, LinearExpr};
    use cp_sat::proto::CpSolverStatus;

    let jobs_data: Vec<Vec<(i64, i64)>> = vec![
        vec![(0, 3), (1, 2), (2, 2)],
        vec![(0, 2), (2, 1), (1, 4)],
        vec![(1, 4), (2, 3)],
    ];
    let deadlines: Vec<i64> = vec![10, 12, 8];
    let weight_makespan: i64 = 10;
    let weight_total_tardiness: i64 = 1;

    let mut model = CpModelBuilder::default();
    let horizon: i64 = jobs_data
        .iter()
        .flat_map(|job| job.iter().map(|(_, d)| *d))
        .sum();
    let starts: Vec<Vec<IntVar>> = jobs_data
        .iter()
        .map(|job| (0..job.len()).map(|_| model.new_int_var([(0, horizon)])).collect())
        .collect();
    for (job_id, job) in jobs_data.iter().enumerate() {
        for t in 0..job.len() - 1 {
            model.add_ge(starts[job_id][t + 1], LinearExpr::from(starts[job_id][t]) + job[t].1);
        }
    }
    let num_machines =
        jobs_data.iter().flat_map(|job| job.iter().map(|(m, _)| *m)).max().unwrap_or(0) + 1;
    for m in 0..num_machines {
        let mut ops = Vec::new();
        for (job_id, job) in jobs_data.iter().enumerate() {
            for (t, (om, d)) in job.iter().enumerate() {
                if *om == m {
                    ops.push((job_id, t, *d));
                }
            }
        }
        for a in 0..ops.len() {
            for b in (a + 1)..ops.len() {
                let (ja, ta, da) = ops[a];
                let (jb, tb, db) = ops[b];
                let before = model.new_bool_var();
                let c1 = model.add_linear_constraint(
                    [(1, starts[jb][tb]), (-1, starts[ja][ta])],
                    [(da, i64::MAX)],
                );
                model.only_enforce_if(c1, [before]);
                let c2 = model.add_linear_constraint(
                    [(1, starts[ja][ta]), (-1, starts[jb][tb])],
                    [(db, i64::MAX)],
                );
                model.only_enforce_if(c2, [!before]);
            }
        }
    }

    // Makespan = max des fins
    let ends: Vec<LinearExpr> = jobs_data
        .iter()
        .enumerate()
        .map(|(j, job)| {
            let last = job.len() - 1;
            LinearExpr::from(starts[j][last]) + job[last].1
        })
        .collect();
    let makespan = model.new_int_var([(0, horizon)]);
    model.add_max_eq(makespan, ends);

    // Tardiness par job : tard_j >= C_j - d_j ET tard_j >= 0, minimiser les
    // contraint. Ici : tard_j - debut_derniere_op >= duree - d_j (ligneaire).
    let mut tardiness_vars: Vec<IntVar> = Vec::new();
    for (j, job) in jobs_data.iter().enumerate() {
        let last = job.len() - 1;
        let tard = model.new_int_var([(0, horizon)]);
        model.add_linear_constraint(
            [(1, tard), (-1, starts[j][last])],
            [(job[last].1 - deadlines[j], i64::MAX)],
        );
        tardiness_vars.push(tard);
    }
    let total_tardiness = model.new_int_var([(0, horizon * jobs_data.len() as i64)]);
    let tard_sum: LinearExpr = tardiness_vars
        .iter()
        .fold(LinearExpr::from(0), |acc, t| acc + LinearExpr::from(*t));
    model.add_eq(total_tardiness, tard_sum);

    // Objectif pondere : 10 * makespan + 1 * total_tardiness
    model.minimize(
        LinearExpr::from(makespan) * weight_makespan
            + LinearExpr::from(total_tardiness) * weight_total_tardiness,
    );

    let response = model.solve();
    let status = response.status();
    let label = match status {
        CpSolverStatus::Optimal => "OPTIMAL",
        CpSolverStatus::Feasible => "FEASIBLE",
        _ => "INFEASIBLE",
    };
    let per_job: Vec<i64> = tardiness_vars
        .iter()
        .map(|v| v.solution_value(&response))
        .collect();
    println!("Exercice 4 - JSSP multi-objectif");
    println!("Status: {}", label);
    println!("Makespan: {}", makespan.solution_value(&response));
    println!("Total tardiness: {}", total_tardiness.solution_value(&response));
    println!("Tardiness per job: {:?}", per_job);
    println!("Objective value: {:.0}", response.objective_value);
}
let _ = ();

Exercice 4 - JSSP multi-objectif


Status: OPTIMAL


Makespan: 11


Total tardiness: 0


Tardiness per job: [0, 0, 0]


Objective value: 110


### Exercice 4 : Exploration du front de Pareto

**Enonce** : Au lieu d'utiliser une ponderation fixe, trouvez plusieurs solutions Pareto-optimales en variant les poids `weight_makespan` et `weight_tardiness`.

**Consignes** :
1. Testez au moins 5 combinaisons de poids (ex: (10,1), (5,5), (1,10), (1,1), (20,1))
2. Pour chaque combinaison, affichez makespan, tardiness totale et objectif pondere
3. Tracez le front de Pareto (makespan vs tardiness) — en Rust, une table ASCII des couples suffit

In [19]:
// Exercice 4 : Exploration du front de Pareto

// TODO: bouclez sur differentes ponderations et collectez les resultats
// Indice : stockez (makespan, tardiness) dans un Vec et affichez un tableau

// Votre code ici
println!("Exercice a completer");

Exercice a completer


## Références

1. **OR-Tools Documentation**: https://developers.google.com/optimization/scheduling
2. **Handbook of Scheduling** (2004): J.Y.-T. Leung ed.
3. **Constraint-Based Scheduling** (2001): P. Baptiste, C. Le Pape, W. Nuijten
4. **CP-SAT Primer** : [d-krupke.github.io/cpsat-primer](https://d-krupke.github.io/cpsat-primer/) (Krupke & Schmitt, 2023, CC-BY) ; documentation OR-Tools : [ortools/sat/docs](https://github.com/google/or-tools/tree/stable/ortools/sat/docs)

5. Garey, M.R. & Johnson, D.S. (1979). Computers and Intractability: A Guide to the Theory of NP-Completeness. W.H. Freeman.
6. Brucker, P. (2007). Scheduling Algorithms (5th ed.). Springer.
7. Herroelen, W., De Reyck, B. & Demeulemeester, E. (1998). Resource-Constrained Project Scheduling: A Survey of Recent Developments. Computers & Opérations Research 25(4):279-302.
8. Burke, E.K., De Causmaecker, P., Vanden Berghe, G. & Van Landeghem, H. (2004). The State of the Art of Nurse Rostering. Journal of Scheduling 7(6):441-499.

Crate Rust : [cp_sat 0.4.1](https://crates.io/crates/cp_sat) — bindings communautaires du moteur CP-SAT, liées à OR-Tools C++ natif.

***

## Conclusion

Ce notebook a appliqué les CSP aux **problèmes d'ordonnancement**, omniprésents dans l'industrie, sur le moteur CP-SAT via la crate Rust `cp_sat`.

### Concepts clés

| Concept | Description |
|---------|-------------|
| **Job Shop Scheduling** | Ordonnancer des tâches sur des machines avec contraintes de précédence |
| **Ressource disjonctive** | Une machine ne peut traiter qu'une tâche à la fois |
| **Contraintes temporelles** | Durées, dates de début/fin, délais |
| **Makespan** | Temps total pour compléter toutes les tâches |
| **Contraintes globales** | Cumulative, all-différent pour CSP complexes |

### Modélisation CSP du scheduling

| Élément | Modélisation CSP |
|---------|------------------|
| **Tâches** | Variables (date de début) |
| **Machines** | Contraintes de disjonction |
| **Précédence** | Contraintes d'inégalité |
| **Objectif** | Minimiser le makespan |

### Écarts d'API Rust assumés (vs carnet Python)

L'API de la crate `cp_sat` 0.4.1 est plus fine que le module Python `ortools.sat.python.cp_model` ; ce portage documente chaque écart, sans jamais fabriquer de sortie :

| Python OR-Tools | Portage Rust | Nature |
|---|---|---|
| `AddNoOverlap` | paires réifiées par booléen (exact) | décomposition |
| `AddCumulative` | booléens de pas `s/f` temporels (exact) | décomposition |
| `AddDecisionStrategy` | non mesuré (2 configs au lieu de 3) | écart assumé |
| callback de solution | paliers de budget 1/2/4/6 s | substitut |
| courbes matplotlib | tables + barres ASCII | substitut |

### Points clés à retenir

1. Le **Job Shop Scheduling** est NP-difficile, mais CSP le rend tractable
2. La **modélisation par contraintes** est plus expressive que la programmation linéaire
3. Les **contraintes globales** (cumulative) capturent des patterns complexes — et se décomposent exactement quand l'API ne les expose pas
4. La **recherche locale** est souvent utilisée pour des problèmes réels
5. Les solveurs modernes (CP-SAT, OR-Tools) dominent les approches pures

**Voir aussi** :
- [CSP-04-Scheduling-Python.ipynb](CSP-04-Scheduling-Python.ipynb) — le carnet Python source de ce portage
- [CSP-05-Optimization-Python.ipynb](CSP-05-Optimization-Python.ipynb) pour les problèmes d'optimisation
- [App-04-JobShopScheduling-Python.ipynb](../Part3-Cas-pratiques/App-04-JobShopScheduling-Python.ipynb) - Application détaillée